#pyspark intro 

a short intro on notbooks in databricks- can choose between 
- sql notebook 
- pythone notbook 


the choise makes all the cells default to the choise e.g SQL -> cells become SQL by defaulet 



In [0]:
DATA_PATH = "/Volumes/data/olympic_games/raw_data"

df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True)
df_athletes 

                             

In [0]:
df_athletes.limit(2).display()

In [0]:
df_athletes.printSchema()

In [0]:
df_athletes.count(), len(df_athletes.columns)


# infert the schema 

In [0]:
df_athletes = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True , inferSchema=True )
df_athletes 


In [0]:
display(df_athletes)

# defind explicit shcema 


In [0]:
from pyspark.sql.types import StructField, StringType, ShortType, ByteType, StructType, IntegerType, FloatType

schema = StructType([
  StructField("ID", IntegerType()),
  StructField("Name", StringType()),
  StructField("Sex", StringType()),
  StructField("Age", ByteType()),
  StructField("Height", ShortType()),
  StructField("Weight", ShortType()),
  StructField("Team", StringType()),
  StructField("NOC", StringType()),
  StructField("Games", StringType()),
  StructField("Year", ShortType()),
  StructField("Season", StringType()),
  StructField("City", StringType()),
  StructField("Sport", StringType()),
  StructField("Event", StringType()),
  StructField("Medal", StringType())
])



df_athletes_schema = spark.read.csv(f"{DATA_PATH}/athlete_events.csv", header=True, schema=schema)
display(df_athletes_schema)

# EDA 


In [0]:

df_athletes_schema.groupBy("NOC", "Medal").count().filter(
    "NOC IN ('SWE', 'NOR', 'FIN', 'DEN', 'ISL') AND Medal != 'NA'"
).sort("NOC", "Medal").display()

In [0]:

df_athletes_schema.createOrReplaceTempView("df_athletes_schema")

# TODO: pick out swedish medals in table tennis
spark.sql("""
          SELECT
            DISTINCT sport
          FROM df_athletes_schema
          
          """).display()


In [0]:
spark.sql(
    """
          SELECT
            name, 
            sport,
            event,
            medal,
            year
          FROM df_athletes_schema
          WHERE 
            sport = 'Table Tennis' AND
            medal != 'NA' AND
            noc = 'SWE'
          """
).display()


#Count medals and plot 


In [0]:
df_swe_medals = spark.sql("""
          SELECT
            sport, count(medal) AS medal_count
          FROM df_athletes_schema
          WHERE noc = 'SWE' AND medal IN ('Bronze', 'Silver','Gold')
          GROUP BY sport 
          ORDER BY medal_count DESC
          """)

df_swe_medals.display()

Databricks visualization. Run in Databricks to view.

In [0]:
df_swe_medals.plot(kind= "bar", y= "sport", x = "medal_count" )

# ingesting data to unity catalog

In [0]:
%sql 
SHOW SCHEMAS IN data;

create table if not EXISTS data.olympic_games.sweden_medals as 
(   
    select 
        name,
        age,
        height,
        weight,
        sport,
        year,
        medal
    from df_athletes_schema 
    where noc = 'SWE' and medal in ('Gold','Silver','Bronze')
)
